# 10 — RQ1 đánh giá cuối: chấm điểm Hold-out, focused set và Lazada (máy local)

**Lần mở duy nhất**

Đây là nơi duy nhất đọc nhãn gold của Hold-out (1.000), focused set (150) và Lazada (500). Notebook chỉ chạy sau khi 10a (mô hình
neural, đã qua cổng Bước 0) và 10b (LLM) đã lưu đủ dự đoán. Không có bước tinh chỉnh nào; cấu hình và ngưỡng đọc từ
`03_configs/frozen/` (amendment v2.1).

**Quy ước báo cáo**

Mô hình neural được báo cáo bằng trung bình ± độ lệch chuẩn qua ba seed; seed 42 là lần chạy chuẩn (Q6) dùng cho bootstrap, ma trận
nhầm lẫn và hybrid 3×3. LLM là một lần chạy. Khoảng tin cậy 95% dùng bootstrap 10.000 lần (seed 42) với cùng các mẫu lại cho mọi
cấu hình, nên chênh lệch giữa hai cấu hình là chênh lệch ghép cặp; hai cấu hình được xem là tương đương khi CI của chênh lệch chứa 0.

**Không kết luận trong notebook**

Notebook in bảng và khoảng tin cậy; phần diễn giải được viết sau khi xem kết quả.

In [ ]:
import os, sys, json, subprocess, datetime
import numpy as np, pandas as pd
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification import evaluate as E
from src.classification.tier2_bfu.mapping import TIER2_LABELS
from src.classification.final_eval import (LOCKED, load_frozen, neural_probs, llm_ids, pm, point_metric, bootstrap_table,
                                           hybrid_predictions)
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
FZ = load_frozen(ROOT)
SEEDS = FZ['tier1']['seeds']
OPEN_DATE = datetime.date.today().strftime('%d/%m/%Y')
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
DIRTY = bool(subprocess.run(['git', '-C', ROOT, 'status', '--porcelain'], capture_output=True, text=True).stdout.strip())
print('Ngày mở Hold-out / focused / Lazada:', OPEN_DATE, '| commit:', COMMIT, '| có thay đổi chưa commit:', DIRTY)
RUN = json.load(open(os.path.join(RES, 'gd10_run_info.json'), encoding='utf-8'))
print('10a: ngày', RUN['run_date'], '| commit', RUN['commit'], '| cổng Bước 0 đạt:', RUN['gate_pass'])
assert RUN['gate_pass'], 'Cổng Bước 0 của 10a không đạt — không chấm điểm.'
NEED = [f'gd10_pred_{s}.csv' for s in LOCKED] + [f'gd10_llm_{s}_tier{t}.csv' for s in ('holdout', 'lazada') for t in (1, 2)] \
       + ['gd10_llm_focused_tier2.csv', 'gd10_costs_neural.csv', 'gd10_llm_costs.csv']
missing = [f for f in NEED if not os.path.exists(os.path.join(RES, f))]
assert not missing, f'Thiếu file: {missing}'

## Dữ liệu và dự đoán

**Nhãn gold**

Đọc `gold_label` của ba tập khóa, quy về id bốn lớp BUG=0, FEATURE=1, UX=2, OTHER=3.

**Dự đoán theo chiến lược**

Tầng 1 của neural lấy xác suất ACTIONABLE với ngưỡng đóng băng (ViSoBERT 0,11; Bi-LSTM 0,09); tầng 2 và baseline lấy argmax. LLM
dùng nhãn trả về; tầng 2 của LLM có giá trị −1 ở review không được gọi, và mọi chỗ cần dùng đều được kiểm tra đủ phủ.

In [ ]:
FOUR = E.FOUR_CLASSES
GOLD = {s: pd.read_csv(os.path.join(DATA, f), usecols=['review_id', 'gold_label'], encoding='utf-8-sig') for s, f in LOCKED.items()}
IDS = {s: GOLD[s]['review_id'].tolist() for s in LOCKED}
Y4 = {s: GOLD[s]['gold_label'].map({l: i for i, l in enumerate(FOUR)}).to_numpy() for s in LOCKED}
PRED = {s: pd.read_csv(os.path.join(RES, f'gd10_pred_{s}.csv')) for s in LOCKED}
LLM = {}
for s, t in (('holdout', 1), ('holdout', 2), ('lazada', 1), ('lazada', 2), ('focused', 2)):
    LLM[(s, t)] = pd.read_csv(os.path.join(RES, f'gd10_llm_{s}_tier{t}.csv'))
NEURAL = {'ViSoBERT': 'visobert', 'Bi-LSTM + fastText': 'bilstm'}
STRATS = list(NEURAL) + ['LLM few-shot']

def tier1_act(s):
    out = {}
    for name, m in NEURAL.items():
        thr = FZ['tier1'][m]['threshold']
        out[name] = {sd: p[:, 1] >= thr for sd, p in neural_probs(PRED[s], IDS[s], 'tier1', m, SEEDS).items()}
    if (s, 1) in LLM:
        ids = llm_ids(IDS[s], LLM[(s, 1)])
        assert (ids >= 0).all(), f'{s}: LLM tầng 1 thiếu review'
        out['LLM few-shot'] = {'llm': ids == 1}
    return out

def tier2_ids(s):
    out = {name: {sd: p.argmax(1) for sd, p in neural_probs(PRED[s], IDS[s], 'tier2', m, SEEDS).items()}
           for name, m in NEURAL.items()}
    out['LLM few-shot'] = {'llm': llm_ids(IDS[s], LLM[(s, 2)])}
    return out

def flat_ids(s):
    return {sd: p.argmax(1) for sd, p in neural_probs(PRED[s], IDS[s], 'flat', 'visobert', SEEDS).items()}

def canon(runs):
    return runs[42] if 42 in runs else runs['llm']

for s in LOCKED:
    print(f'{s}: {len(IDS[s])} review | các lớp có mặt: {sorted(set(FOUR[i] for i in Y4[s]))}')

## Tầng 1 trên Hold-out

**Chỉ số**

F1, Recall và Precision của ACTIONABLE ở ngưỡng đóng băng, Macro-F1 hai lớp; neural là trung bình ± độ lệch chuẩn qua ba seed.
Khoảng tin cậy và chênh lệch ghép cặp tính trên seed 42 và lần chạy LLM.

In [ ]:
s = 'holdout'
y1 = (Y4[s] != 3).astype(int)
A1 = tier1_act(s)
rows = []
for name, runs in A1.items():
    v = {met: [point_metric(y1, a.astype(int), 2, met, None if met == 'macro_f1' else 1) for a in runs.values()]
         for met in ('f1', 'recall', 'precision', 'macro_f1')}
    rows.append({'chiến lược': name, 'F1 ACT': pm(v['f1']), 'Recall ACT': pm(v['recall']), 'Precision ACT': pm(v['precision']),
                 'Macro-F1 (2 lớp)': pm(v['macro_f1']), 'số lần chạy': len(runs)})
T1 = pd.DataFrame(rows)
print('Tầng 1 — Hold-out (n = %d; %%):' % len(y1)); print(T1.to_string(index=False))
P1 = {name: canon(runs).astype(int) for name, runs in A1.items()}
T1_CI = []
for met, cls in (('f1', 1), ('recall', 1), ('macro_f1', None)):
    ci, diff = bootstrap_table(y1, P1, 2, met, cls)
    ci.insert(0, 'chỉ số', met); diff.insert(0, 'chỉ số', met)
    T1_CI.append((ci, diff))
    print(f'\n{met} — CI 95% (seed 42 / LLM):'); print(ci.round(4).to_string(index=False)); print(diff.round(4).to_string(index=False))

## Tầng 2 trên gold-actionable của Hold-out

**Chỉ số**

Macro-F1 ba lớp và F1 từng lớp trên các review có nhãn gold BUG/FEATURE/UX, tách khỏi lỗi của tầng 1.

In [ ]:
def tier2_table(s, title):
    mask = Y4[s] != 3
    y2 = Y4[s][mask]
    T2r = tier2_ids(s)
    rows, preds = [], {}
    for name, runs in T2r.items():
        ms = []
        for p in runs.values():
            p = p[mask]
            assert (p >= 0).all(), f'{s}/{name}: tầng 2 thiếu dự đoán trên gold-actionable'
            ms.append(E.metrics(y2, p, TIER2_LABELS))
        rows.append({'chiến lược': name, 'Macro-F1': pm([m['macro_f1'] for m in ms]),
                     **{f'F1 {c}': pm([m[f'f1_{c}'] for m in ms]) for c in TIER2_LABELS}, 'số lần chạy': len(runs)})
        preds[name] = canon(runs)[mask]
    tab = pd.DataFrame(rows)
    print(f'{title} (n = {mask.sum()}; %):'); print(tab.to_string(index=False))
    ci_m, diff_m = bootstrap_table(y2, preds, 3)
    ci_f, diff_f = bootstrap_table(y2, preds, 3, 'f1', 1)
    print('\nMacro-F1 — CI 95%:'); print(ci_m.round(4).to_string(index=False)); print(diff_m.round(4).to_string(index=False))
    print('\nF1 FEATURE — CI 95%:'); print(ci_f.round(4).to_string(index=False)); print(diff_f.round(4).to_string(index=False))
    return tab, ci_m, diff_m, ci_f, diff_f

T2 = tier2_table('holdout', 'Tầng 2 — Hold-out, gold-actionable')

## End-to-end bốn lớp trên Hold-out: cascade và baseline phẳng

**Cách ghép**

Cascade của mỗi chiến lược: tầng 1 gọi NOT_ACTIONABLE thì nhãn OTHER, ngược lại lấy nhãn tầng 2 của cùng chiến lược và cùng seed.
Baseline phẳng là ViSoBERT bốn lớp.

**Chỉ số**

Macro-F1 bốn lớp, F1 từng lớp, khoảng tin cậy, chênh lệch ghép cặp và ma trận nhầm lẫn (seed 42 / LLM).

In [ ]:
def cascade_runs(s):
    A, T = tier1_act(s), tier2_ids(s)
    out = {}
    for name in STRATS:
        if name not in A:
            continue
        runs = {}
        for key, act in A[name].items():
            t2 = T[name][key]
            assert not ((t2 < 0) & act).any(), f'{s}/{name}: tầng 2 thiếu dự đoán cho review tầng 1 giữ lại'
            runs[key] = E.cascade_labels(act, np.where(t2 < 0, 0, t2))
        out[f'Cascade {name}'] = runs
    return out

def e2e_table(s, runs_by_name, title):
    rows = []
    for name, runs in runs_by_name.items():
        ms = [E.metrics(Y4[s], p, FOUR) for p in runs.values()]
        rows.append({'cấu hình': name, 'Macro-F1 (4 lớp)': pm([m['macro_f1'] for m in ms]),
                     **{f'F1 {c}': pm([m[f'f1_{c}'] for m in ms]) for c in FOUR}, 'số lần chạy': len(runs)})
    tab = pd.DataFrame(rows)
    print(f'{title} (n = {len(Y4[s])}; %):'); print(tab.to_string(index=False))
    ci, diff = bootstrap_table(Y4[s], {n: canon(r) for n, r in runs_by_name.items()}, 4)
    print('\nMacro-F1 bốn lớp — CI 95%:'); print(ci.round(4).to_string(index=False)); print(diff.round(4).to_string(index=False))
    return tab, ci, diff

E2E_RUNS = {**cascade_runs('holdout'), 'Baseline phẳng ViSoBERT': flat_ids('holdout')}
E2E = e2e_table('holdout', E2E_RUNS, 'End-to-end — Hold-out')
CONF = {}
for name, runs in E2E_RUNS.items():
    CONF[name] = E.confusion(Y4['holdout'], canon(runs), FOUR)
    print(f'\nMa trận nhầm lẫn — {name} (seed 42 / LLM):'); print(CONF[name].to_string())

## Focused FEATURE/UX set (báo cáo tách biệt)

**Tính chất của tập**

Focused set gồm 150 review được chọn theo cụm từ gợi ý FEATURE/UX, nên là tập làm giàu, không phản ánh phân bố tự nhiên. Tập chỉ dùng
để đo khả năng phân biệt lớp hiếm ở tầng 2; kết quả không gộp vào chỉ số chính.

In [ ]:
FOC = tier2_table('focused', 'Tầng 2 — focused set (tập làm giàu), gold-actionable')

## External Test Lazada

**Mục đích**

Kiểm tra khả năng khái quát sang ứng dụng khác bằng cascade end-to-end của ba chiến lược, với mô hình và prompt đã đóng băng, không
tinh chỉnh lại. Lazada có phân bố khác Shopee (1★ khoảng 60%).

In [ ]:
LAZ = e2e_table('lazada', cascade_runs('lazada'), 'End-to-end — Lazada')

## Hybrid 3×3 trên Hold-out

**Cách ghép**

Tầng 1 của chiến lược X ghép với tầng 2 của chiến lược Y (seed 42 / LLM), từ dự đoán đã lưu, không train lại. Đường chéo là ba cascade
thuần.

**Chi phí của mỗi tổ hợp**

Thời gian và chi phí API cho 1.000 review = chi phí tầng 1 trên toàn bộ review + chi phí tầng 2 trên phần tầng 1 giữ lại (tỷ lệ giữ
lại đo trên Hold-out).

In [ ]:
A, T = tier1_act('holdout'), tier2_ids('holdout')
HY = hybrid_predictions({x: canon(A[x]) for x in STRATS}, {y: canon(T[y]) for y in STRATS})
HY_CI, _ = bootstrap_table(Y4['holdout'], {f'{x} × {y}': p for (x, y), p in HY.items()}, 4)
CN = pd.read_csv(os.path.join(RES, 'gd10_costs_neural.csv')).set_index(['tier', 'model'])
CL = pd.read_csv(os.path.join(RES, 'gd10_llm_costs.csv')).set_index('bước')
SEC = {1: {n: CN.loc[('tier1', m), 'infer_seconds_per_1000_mean'] for n, m in NEURAL.items()},
       2: {n: CN.loc[('tier2', m), 'infer_seconds_per_1000_mean'] for n, m in NEURAL.items()}}
USD = {1: {n: 0.0 for n in NEURAL}, 2: {n: 0.0 for n in NEURAL}}
SEC[1]['LLM few-shot'], USD[1]['LLM few-shot'] = CL.loc['holdout – tầng 1', 'giây_/1000_review'], CL.loc['holdout – tầng 1', 'usd_/1000_review']
SEC[2]['LLM few-shot'], USD[2]['LLM few-shot'] = CL.loc['holdout – tầng 2', 'giây_/1000_review'], CL.loc['holdout – tầng 2', 'usd_/1000_review']
rows = []
for (x, y), p in HY.items():
    keep = float(canon(A[x]).mean())
    ci = HY_CI.set_index('cấu hình').loc[f'{x} × {y}']
    rows.append({'tầng 1 (X)': x, 'tầng 2 (Y)': y, 'thuần': x == y, 'Macro-F1 (4 lớp)': round(ci['giá trị'] * 100, 1),
                 'CI 95%': f"{ci['CI 2,5%'] * 100:.1f}–{ci['CI 97,5%'] * 100:.1f}", 'tỷ lệ tầng 1 giữ lại': round(keep, 3),
                 'giây / 1.000 review': round(SEC[1][x] + keep * SEC[2][y], 2),
                 'USD API / 1.000 review': round(USD[1][x] + keep * USD[2][y], 4)})
HYB = pd.DataFrame(rows)
print('Hybrid 3×3 — Hold-out (seed 42 / LLM):'); print(HYB.to_string(index=False))
print(); print(HYB.pivot(index='tầng 1 (X)', columns='tầng 2 (Y)', values='Macro-F1 (4 lớp)').loc[STRATS, STRATS].to_string())

## Chi phí tổng hợp (cost-aware)

**Các chiều chi phí**

Nhãn cần để xây dựng, thời gian huấn luyện, thời gian suy luận cho 1.000 review (cascade = tầng 1 + tầng 2 trên phần giữ lại), bộ nhớ
GPU, chi phí API thực và mức phụ thuộc nhà cung cấp. Thời gian gán nhãn thực tế chưa được ghi nhận và để trống.

In [ ]:
keep = {n: float(canon(A[n]).mean()) for n in STRATS}
COST = []
for n, m in NEURAL.items():
    COST.append({'chiến lược': n, 'nhãn cần': 'Train 1.200 + Validation 350',
                 'huấn luyện / seed (giây, tầng 1 + tầng 2)': round(CN.loc[('tier1', m), 'train_seconds_mean'] + CN.loc[('tier2', m), 'train_seconds_mean'], 1),
                 'suy luận / 1.000 review (giây)': round(SEC[1][n] + keep[n] * SEC[2][n], 2),
                 'GPU tối đa (GB)': max(CN.loc[('tier1', m), 'peak_gpu_gb_max'], CN.loc[('tier2', m), 'peak_gpu_gb_max']),
                 'USD API / 1.000 review': 0.0, 'phụ thuộc nhà cung cấp': 'Không (chạy cục bộ)'})
COST.append({'chiến lược': 'LLM few-shot', 'nhãn cần': '17 ví dụ (Train) + Validation 350', 'huấn luyện / seed (giây, tầng 1 + tầng 2)': 0.0,
             'suy luận / 1.000 review (giây)': round(SEC[1]['LLM few-shot'] + keep['LLM few-shot'] * SEC[2]['LLM few-shot'], 1),
             'GPU tối đa (GB)': 0.0, 'USD API / 1.000 review': round(USD[1]['LLM few-shot'] + keep['LLM few-shot'] * USD[2]['LLM few-shot'], 4),
             'phụ thuộc nhà cung cấp': 'Có (OpenAI, snapshot cố định)'})
COST.append({'chiến lược': 'Baseline phẳng ViSoBERT', 'nhãn cần': 'Train 1.200 + Validation 350',
             'huấn luyện / seed (giây, tầng 1 + tầng 2)': round(CN.loc[('flat', 'visobert'), 'train_seconds_mean'], 1),
             'suy luận / 1.000 review (giây)': round(CN.loc[('flat', 'visobert'), 'infer_seconds_per_1000_mean'], 2),
             'GPU tối đa (GB)': CN.loc[('flat', 'visobert'), 'peak_gpu_gb_max'], 'USD API / 1.000 review': 0.0,
             'phụ thuộc nhà cung cấp': 'Không (chạy cục bộ)'})
COST = pd.DataFrame(COST)
COST['giờ gán nhãn'] = '{...}'
print(COST.to_string(index=False))
print('GPU khi train lại:', sorted(set(CN['gpu'])))

## Lưu kết quả

**Các file**

Mọi bảng được lưu với tiền tố `gd10_` trong `05_experiments/rq1_classification/`. Ngày mở và commit in ở đầu notebook dùng để cập nhật
`01_data/HOLDOUT_LOCK.md`.

In [ ]:
def save(df, name):
    df.to_csv(os.path.join(RES, f'gd10_{name}.csv'), index=False)

save(T1, 'tier1_holdout'); save(pd.concat([c for c, _ in T1_CI]), 'tier1_holdout_ci'); save(pd.concat([d for _, d in T1_CI]), 'tier1_holdout_diff')
save(T2[0], 'tier2_holdout'); save(T2[1], 'tier2_holdout_ci'); save(T2[2], 'tier2_holdout_diff'); save(T2[3], 'tier2_holdout_feature_ci')
save(E2E[0], 'e2e_holdout'); save(E2E[1], 'e2e_holdout_ci'); save(E2E[2], 'e2e_holdout_diff')
pd.concat([c.assign(cấu_hình=n) for n, c in CONF.items()]).to_csv(os.path.join(RES, 'gd10_confusion_holdout.csv'))
save(FOC[0], 'tier2_focused'); save(FOC[1], 'tier2_focused_ci'); save(FOC[3], 'tier2_focused_feature_ci')
save(LAZ[0], 'e2e_lazada'); save(LAZ[1], 'e2e_lazada_ci'); save(LAZ[2], 'e2e_lazada_diff')
save(HYB, 'hybrid_holdout'); save(COST, 'cost_summary')
json.dump({'open_date': OPEN_DATE, 'commit': COMMIT, 'uncommitted_changes': DIRTY}, open(os.path.join(RES, 'gd10_open_info.json'), 'w'), indent=1)
print('Đã lưu:', sorted(f for f in os.listdir(RES) if f.startswith('gd10_')))

## Nhận xét

**Chưa viết**

Phần nhận xét được viết sau khi xem các bảng và khoảng tin cậy ở trên; notebook không đưa ra kết luận chiến lược nào phù hợp nhất.